# Does weaker acid change what survives? Test one small peptide at two acid strengths

You'll pick one small peptide and decide whether it survives in 98% and in 81% sulfuric acid.

The study behind notebook 01 tested all 20 homodipeptides (two copies of one amino acid joined,
such as Ala-Ala) in both acids. Once reviewed, your verdict goes into the community replication
table.

| Part | What you do | Time |
|---|---|---|
| 0 | Pick a dipeptide and predict | 10 min |
| 1 | Stack both series and check them | 20 min |
| 2 | Compare first and last in each acid | 25 min |
| 3 | Write your verdict | 10 min |
| 4 | Check your verdict against the paper | 5 min |

There is no worked example for your molecule. You run the full analysis yourself.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0. Paper: *Scientific Reports* 14, 17083 (2024).

In [ ]:
# Setup. In Colab this first installs the training package (about 30 seconds).
import importlib.util, os
if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
import matplotlib.pyplot as plt
import nmrtrain as nt

NOTEBOOK = "notebooks/02_one_dipeptide_both_acids.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

## Part 0: Pick and predict

In [ ]:
for code_ in nt.data.samples("11223995"):
    print(code_, nt.io.NAMES[code_])

**Your turn**

Pick a dipeptide from the list above, or use the one your mentor assigned. In the next cell:

* Set `DIPEPTIDE` to its code.
* Predict what happens to it in 98% acid (`"K"`) and in 81% acid (`"H"`): `"survives"`, `"degrades"` or `"unsure"`.
* Say which acid you expect to be harsher on it.

In [ ]:
DIPEPTIDE = "LL"   # change this to the code of the dipeptide you picked

MY_PREDICTION = {
    "K": ...,          # 98% acid
    "H": ...,          # 81% acid
    "harsher": ...,    # which acid is harsher on it: "98%", "81%", "same" or "unsure"?
}
nt.record.predict("dipeptide", MY_PREDICTION)

## Part 1: Load and stack

Acid code `K` is 98% and `H` is 81%. The next cell downloads both series (about 5 MB) and prints
where the acid peak sits in each spectrum.

In [ ]:
series = {acid: nt.load_series("11223995", DIPEPTIDE, acid=acid, nucleus="1H") for acid in "KH"}
for acid, ser in series.items():
    print(acid, [s.time_label for s in ser], "| acid peak at", [round(nt.checks.acid_ppm(s), 2) for s in ser])

In 81% acid the acid peak sits higher (about 11.7 to 12.1 ppm), and your molecule's peaks sit at
different ppm than in 98%. Compare each acid's series only with itself, never across acids.

**Your turn**

Set `REGION` to the ppm range you will read, for example `(1.0, 6.5)`. Go wider, for example
`(0.5, 9.5)`, if your molecule has ring protons (they sit between 6.5 and 9 ppm). Left as `...`,
the cell uses `(0.5, 9.5)`.

In [ ]:
REGION = ...   # for example (1.0, 6.5) or (0.5, 9.5)
REGION = nt.record.fill(REGION, (0.5, 9.5), "REGION")

fig_stacks, axes = plt.subplots(1, 2, figsize=(14, 5.5))
traps = {}
for ax, (acid, ser) in zip(axes, series.items()):
    nt.plot.stack(ser, REGION[0], REGION[1], norm=REGION, ax=ax,
                  title=f"{nt.io.NAMES[DIPEPTIDE]} in {nt.io.ACID_PCT[acid]} acid")
    traps[acid] = nt.checks.run_traps(ser, norm=REGION, lo=REGION[0], hi=REGION[1])
plt.show()
nt.portal.done("dipeptide-stack", figures=[fig_stacks]);

## Part 2: First vs last, in each acid

In some samples the whole spectrum moves a little between the first and last time points,
especially in 81% acid. A peak that moved by the same amount as all the others is the same peak.
The next cell lists the peaks kept, lost and new in each acid.

In [ ]:
MIN_HEIGHT = 0.03
my_peaks, fig_cmp = {}, plt.figure(figsize=(14, 5))
for i, (acid, ser) in enumerate(series.items()):
    first, last = ser[0], ser[-1]
    my_peaks[acid] = {first.time: nt.peaks.pick(first, *REGION, MIN_HEIGHT),
                      last.time: nt.peaks.pick(last, *REGION, MIN_HEIGHT)}
    ax = fig_cmp.add_subplot(1, 2, i + 1)
    nt.plot.compare(first, last, *REGION, peaks_first=my_peaks[acid][first.time],
                    peaks_last=my_peaks[acid][last.time], ax=ax,
                    title=f"{nt.io.ACID_PCT[acid]}: {first.time_label} vs {last.time_label}")
    print(acid, {k: [round(p, 2) for p in v] for k, v in nt.peaks.compare(*my_peaks[acid].values()).items()})
plt.show()

## Part 3: Your verdict

**Your turn**

Fill in one entry per acid in the next cell, using ppm values from your peak lists in Part 2:

* `call`: `"survives"`, `"degrades"`, `"side chain modified"` or `"unsure"`
* `lost_ppm`: signals gone by the last time point
* `new_ppm`: signals that appear by the last time point
* `kept_ppm`: signals present at both times
* `product_guess`: the main product in words, or `"none"`
* `confidence`: `"low"`, `"medium"` or `"high"`

Also set `harsher` to the acid that was harsher on it. NH signals often fade by exchange with the
acid; that alone is not breakdown.

In [ ]:
MY_VERDICT = {
    "K": {"call": "", "lost_ppm": [], "new_ppm": [], "kept_ppm": [], "product_guess": "", "confidence": ""},
    "H": {"call": "", "lost_ppm": [], "new_ppm": [], "kept_ppm": [], "product_guess": "", "confidence": ""},
    "harsher": "",   # "98%", "81%", "same" or "unsure"
    "note": "",
}
print("\n".join(nt.checks.verdict_problems(MY_VERDICT, ["K", "H"])) or "Verdict format OK.")

## Part 4: Check your verdict against the paper

The paper states a call outright for Ala-Ala and Gly-Gly in 98% acid, and the next cell compares
your call with those. For any other case, check your call against Table 1 (81%) and Table 2 (98%)
of the paper (Petkowski et al. 2024, *Scientific Reports* 14, 17083), and say in your `note` what
you found. The cell does not grade.

In [ ]:
feedback, matches = {}, {}
if nt.record.revealed("dipeptide"):
    key = nt.answers.load("dipeptides")
    for acid in "KH":
        c = nt.answers.case(key, f"{DIPEPTIDE}-{acid}")
        feedback[acid] = nt.checks.check_verdict(MY_VERDICT[acid], c).show(f"Your verdict, {nt.io.ACID_PCT[acid]} acid")
        call = MY_VERDICT[acid]["call"].strip().lower()
        matches[acid] = "unsure" if not c.get("call") or call in ("", "unsure") else ("yes" if call == c["call"] else "no")

## Hand-in

The next cell saves these in `handin/`:

* the stacks for both acids
* the first vs last comparison
* `verdict.json`, with your verdict, peak lists and feedback
* `replication_rows.csv`, your two lines for the replication table (one per acid)

Nothing is sent for review until you run `nt.portal.submit()`.

In [ ]:
n_traps = nt.checks.traps_flagged(*traps.values())
record = nt.record.handin(
    "dipeptide-verdict", figures={"stacks": fig_stacks, "first_vs_last": fig_cmp},
    verdict={"dipeptide": DIPEPTIDE, "verdict": MY_VERDICT,
             "my_peaks_ppm": {a: {t: nt.peaks.ppms(v) for t, v in d.items()} for a, d in my_peaks.items()},
             "feedback": {a: [str(f) for f in v] for a, v in feedback.items()}},
    numbers={"sample": DIPEPTIDE, "call_98": MY_VERDICT["K"]["call"] or "blank",
             "call_81": MY_VERDICT["H"]["call"] or "blank", "harsher": MY_VERDICT["harsher"] or "blank"},
    question="dipeptide", traps=n_traps)
rows = [nt.record.replication_row(DIPEPTIDE, acid, MY_VERDICT[acid], [s.time for s in (series[acid][0], series[acid][-1])],
                                  matches=matches.get(acid, "unsure"), traps=n_traps) for acid in "KH"]
nt.record.save_rows(rows, "dipeptide-verdict");
# nt.portal.submit()